# First steps with PyTorch

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

%reload_ext autoreload
%autoreload 2

print("cwd:", Path.cwd())
print("sys.path:", sys.path[:5])


from src.conformal.pipeline import run_mondrian_regression


cwd: /home/victor/gw/cbc_pe/notebooks
sys.path: ['/home/victor/gw/cbc_pe', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '']


In [2]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

print(torch.cuda.is_available())
print(torch.version.cuda)
print(torch.cuda.device_count())

cuda
True
13.0
1


## Tutorial: How to build a model with CNN

In [3]:
import numpy as np

N = 256
n_detectors = 3
signal_length = 4096

rng = np.random.default_rng(42)

t = np.linspace(0, 1, signal_length, dtype=np.float32)

# Tres patrones temporales simples
pattern_0 = np.sin(2 * np.pi * 20 * t)
pattern_1 = np.sin(2 * np.pi * 40 * t)
pattern_2 = np.sin(2 * np.pi * 80 * t)

patterns = np.stack([pattern_0, pattern_1, pattern_2], axis=0)

# Labels verdaderas artificiales
y_dummy = rng.normal(
    loc=0.0,
    scale=1.0,
    size=(N, 3),
).astype(np.float32)

# Señales: ruido + amplitud * patrón
X_dummy = rng.normal(
    loc=0.0,
    scale=0.2,
    size=(N, n_detectors, signal_length),
).astype(np.float32)

for i in range(N):
    X_dummy[i, 0, :] += y_dummy[i, 0] * patterns[0]
    X_dummy[i, 1, :] += y_dummy[i, 1] * patterns[1]
    X_dummy[i, 2, :] += y_dummy[i, 2] * patterns[2]

# Estandarizar labels
y_mean = y_dummy.mean(axis=0)
y_std = y_dummy.std(axis=0) + 1e-8
y_dummy = (y_dummy - y_mean) / y_std

print(X_dummy.shape)
print(y_dummy.shape)

(256, 3, 4096)
(256, 3)


In [4]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset


class ArrayRegressionDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class TinyCNN(nn.Module):
    def __init__(self, n_detectors=3, n_outputs=3):
        super().__init__()

        self.conv = nn.Conv1d(
            in_channels=n_detectors,
            out_channels=16,
            kernel_size=16,
            stride=2,
            padding=8,
        )

        self.activation = nn.ReLU()
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Linear(16, n_outputs)

    def forward(self, x):
        x = self.conv(x)
        x = self.activation(x)
        x = self.pool(x)
        x = x.squeeze(-1)
        x = self.head(x)
        return x

Divide the data into batches of size 32 and run the model on each batch.

In [5]:
dataset = ArrayRegressionDataset(X=X_dummy, y=y_dummy)

loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
)

X_batch, y_batch = next(iter(loader))

print("X_batch shape:", X_batch.shape)
print("y_batch shape:", y_batch.shape)

X_batch shape: torch.Size([32, 3, 4096])
y_batch shape: torch.Size([32, 3])


In [6]:
model = TinyCNN(n_detectors=3, n_outputs=3)

pred = model(X_batch)

print(pred.shape)

torch.Size([32, 3])


Now we do a training step

In [7]:
epochs = 50

loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(epochs):
    model.train() # Set the model to training mode
    total_loss = 0.0 # Initialize the total loss to zero

    for X_batch, y_batch in loader: # Iterate over the batches
        optimizer.zero_grad() # Reset the gradients to zero before each batch

        pred = model(X_batch) # Compute the predictions using the model
        loss = loss_fn(pred, y_batch) # Compute the loss using the predictions and the ground truth

        loss.backward() # Backpropagate the gradients through the loss
        optimizer.step() # Update the model parameters using the gradients and the optimizer

        total_loss += loss.item() * X_batch.size(0) # Add the loss for the current batch to the total loss

    mean_loss = total_loss / len(dataset) # Compute the mean loss for the current epoch
    if (epoch+1) % 50 == 0: # Print the mean loss every 50 epochs
        print(f"Epoch {epoch+1} of {epochs}: loss = {mean_loss:.6f}") # Print the mean loss for the current epoch


Epoch 50 of 50: loss = 0.775402


## Basic Training Loop

Vamos a implementar los siguientes pasos para entrenar un modelo de CNN:

1. elegir device: CPU/GPU
2. crear Dataset y DataLoader
3. definir modelo
4. definir loss
5. definir optimizer
6. entrenar una epoch
7. validar una epoch
8. hacer un fit loop simple

### 1. Set GPU

In [8]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

print(torch.cuda.is_available())
print(torch.version.cuda)
print(torch.cuda.device_count())

cuda
True
13.0
1


### 2. Model to GPU

In [9]:
# This moves the weights to the GPU if available
model = TinyCNN(n_detectors=3, n_outputs=3)
model = model.to(device) 


### 3. Split train/validation

In [10]:
from torch.utils.data import DataLoader, random_split

dataset = ArrayRegressionDataset(X_dummy, y_dummy)

n_total = len(dataset) # total number of samples
n_train = int(0.8 * n_total) # number of training samples
n_val = n_total - n_train # number of validation samples

train_dataset, val_dataset = random_split(
    dataset, 
    [n_train, n_val],
    generator=torch.Generator().manual_seed(42),
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

### 4. Loss and Optimization

In [11]:
from torch import nn

loss_fn = nn.MSELoss() # It's okay for standarized labels, otherwise the label-scale can produce worse results

optimizer = torch.optim.Adam(
    model.parameters(), 
    lr=1e-2,
)

### 5. Train one-epoch

In [12]:
def train_one_epoch(model, loader, loss_fn, optimizer, device):
    model.train() # Red a modo entrenamiento

    total_loss = 0.0
    n_samples = 0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device) # Mueve los datos a la GPU/CPU
        y_batch = y_batch.to(device)  

        optimizer.zero_grad() # Borra los gradientes y los hace cero

        pred = model(X_batch) # Aplica el modelo y obtiene las predicciones
        loss = loss_fn(pred, y_batch) # Calcula el error de la predicción con la verdad

        loss.backward() # Calcula los gradientes de la función de error
        optimizer.step() # Actualiza los pesos del modelo

        batch_size = X_batch.shape[0]
        total_loss += loss.item() * batch_size
        n_samples += batch_size

    mean_loss = total_loss / n_samples
    return mean_loss
    

### 6. Validate one-epoch

In [13]:
# Desactiva el calculo de gradientes, queremos validar solo, no entrenar
@torch.no_grad() 

def validate_one_epoch(model, loader, loss_fn, device):
    model.eval()

    total_loss = 0.0
    n_samples = 0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        pred = model(X_batch)
        loss = loss_fn(pred, y_batch)

        batch_size = X_batch.size(0)
        total_loss += loss.item() * batch_size
        n_samples += batch_size

    mean_loss = total_loss / n_samples
    return mean_loss

### 7. Fit Loop

In [14]:
num_epochs = 100

for epoch in range(num_epochs):
    train_loss = train_one_epoch(
        model=model,
        loader=train_loader,
        loss_fn=loss_fn,
        optimizer=optimizer,
        device=device,
    )

    val_loss = validate_one_epoch(
        model=model,
        loader=val_loader,
        loss_fn=loss_fn,
        device=device,
    )

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch {epoch+1:03d} | "
            f"train_loss = {train_loss:.6f} | "
            f"val_loss = {val_loss:.6f}"
        )

Epoch 010 | train_loss = 1.015592 | val_loss = 0.921330
Epoch 020 | train_loss = 0.986388 | val_loss = 0.897146
Epoch 030 | train_loss = 0.932269 | val_loss = 0.895993
Epoch 040 | train_loss = 0.859409 | val_loss = 0.795557
Epoch 050 | train_loss = 0.821931 | val_loss = 0.722086
Epoch 060 | train_loss = 0.723952 | val_loss = 0.669297
Epoch 070 | train_loss = 0.664947 | val_loss = 0.628630
Epoch 080 | train_loss = 0.648376 | val_loss = 0.535176
Epoch 090 | train_loss = 0.593530 | val_loss = 0.511017
Epoch 100 | train_loss = 0.549557 | val_loss = 0.473751


# Convolutional Neural Network (CNN): Baseline

Estructura del ConvBlock:
1. **Conv1d**: produce features
2. **BatchNorm1d**: estabiliza las features. Asignando una media de 0 y una varianza de 1.
    - _Entrenamiento_: media y varianza se calculan con los datos de entrenamiento. Usa la estadistica del batch actual.
    - _Evaluacion_: media y varianza se calculan con los datos aprendidos en el entrenamiento. Por eso hay que indicar ``model.eval()`` o ``model.train()``.
3. **ReLU**: activacion lineal


In [15]:


import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset


class ConvBlock(nn.Module):
    
    def __init__(
        self, 
        in_channels, 
        out_channels, 
        kernel_size=16, 
        stride=2, 
    ):
        
        super(ConvBlock, self).__init__() # Initialize parent class nn.Module, otherwise PyTorch will have issues registering the parameters.

        padding = kernel_size // 2

        self.conv = nn.Conv1d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
        )

        self.batch_norm = nn.BatchNorm1d(out_channels)
        self.activation = nn.ReLU()


    def forward(self, x):
        x = self.conv(x)
        x = self.batch_norm(x)
        x = self.activation(x)
        return x

In [16]:
block1 = ConvBlock(
    in_channels=3,
    out_channels=16,
    kernel_size=16,
    stride=2,
)

X_batch, y_batch = next(iter(train_loader))

out = block1(X_batch)

print("Input:", X_batch.shape)
print("Output:", out.shape)

Input: torch.Size([32, 3, 4096])
Output: torch.Size([32, 16, 2049])


Numero de parámetros aprendidos. (``out_channels * in_channels * kernel_size + bias``)

En este ejemplo: ``16 * 3 * 16 + 16 = 784``

Además, si aplicas Batch Normalization, sumas dos parámetros (gamma y beta) por cada canal de entrada. 
(``16 + 16 = 32``)

In [17]:
num_params = sum(p.numel() for p in block1.parameters())
print(num_params)

816


In [18]:
block1 = ConvBlock(
    in_channels=3,
    out_channels=16,
    kernel_size=16,
    stride=2,
)

block2 = ConvBlock(
    in_channels=16,
    out_channels=32,
    kernel_size=16,
    stride=2,
)

pool = nn.AdaptiveAvgPool1d(1) # Adaptive average pooling. Reduce the length to 1.
head = nn.Linear(32, 3)

X_batch, y_batch = next(iter(train_loader))

out1 = block1(X_batch)
out2 = block2(out1)
pooled = pool(out2)
features = pooled.squeeze(-1) # Eliminate the last dimension (1)
y_pred = head(features)

print("Input:       ", X_batch.shape)
print("After block1:", out1.shape)
print("After block2:", out2.shape)
print("After pool:  ", pooled.shape)
print("After head:  ", features.shape)
print("After pred:  ", y_pred.shape)


Input:        torch.Size([32, 3, 4096])
After block1: torch.Size([32, 16, 2049])
After block2: torch.Size([32, 32, 1025])
After pool:   torch.Size([32, 32, 1])
After head:   torch.Size([32, 32])
After pred:   torch.Size([32, 3])


Si queremos pasar a algo que produzca una salida con la predicción de los labels, hay que pasar de:


``(batch_size, n_channels, length) -> (batch_size, n_channels, length)``

Para ello se puede usar un pooling global: ``AdaptativeAvgPool1d(output_size=1)``

In [19]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent # Get the parent root
sys.path.insert(0, str(PROJECT_ROOT)) # include it in sys.path

%reload_ext autoreload
%autoreload 2

print("cwd:", Path.cwd())
print("sys.path:", sys.path[:5])

cwd: /home/victor/gw/cbc_pe/notebooks
sys.path: ['/home/victor/gw/cbc_pe', '/home/victor/gw/cbc_pe', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload']


In [105]:
from src.models.network import SimpleCNN

model = SimpleCNN(
    in_channels=3,
    out_channels=3,
    embedding_dim=64,
)

X_batch, y_batch = next(iter(train_loader))

features = model.encode(X_batch)
embedding = model.embed(X_batch)

pred = model(X_batch) # Esto ya llama internamente a encode y embed (junto con forward)
pred2, emb2 = model(X_batch, return_embedding=True)

print("X:        ", X_batch.shape)
print("features: ", features.shape)
print("embedding:", embedding.shape)
print("pred:     ", pred.shape)
print("pred2:    ", pred2.shape)
print("emb2:     ", emb2.shape)

X:         torch.Size([32, 3, 4096])
features:  torch.Size([32, 32])
embedding: torch.Size([32, 64])
pred:      torch.Size([32, 3])
pred2:     torch.Size([32, 3])
emb2:      torch.Size([32, 64])


In [92]:
num_params = sum(p.numel() for p in model.parameters())
print(num_params)

  #print(model)

11411


In [93]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

model = SimpleCNN(
    in_channels=3,
    out_channels=3,
).to(device)

loss_fn = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
)



cuda


In [103]:
num_epochs = 100

for epoch in range(num_epochs):
    train_loss = train_one_epoch(
        model=model,
        loader=train_loader,
        loss_fn=loss_fn,
        optimizer=optimizer,
        device=device,
    )

    val_loss = validate_one_epoch(
        model=model,
        loader=val_loader,
        loss_fn=loss_fn,
        device=device,
    )

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch {epoch+1:03d} | "
            f"train_loss = {train_loss:.6f} | "
            f"val_loss = {val_loss:.6f}"
        )

Epoch 010 | train_loss = 0.245303 | val_loss = 0.461075
Epoch 020 | train_loss = 0.215451 | val_loss = 0.439576
Epoch 030 | train_loss = 0.207080 | val_loss = 0.476523
Epoch 040 | train_loss = 0.218834 | val_loss = 0.486604
Epoch 050 | train_loss = 0.249494 | val_loss = 0.447969
Epoch 060 | train_loss = 0.204636 | val_loss = 0.496357
Epoch 070 | train_loss = 0.194568 | val_loss = 0.505620
Epoch 080 | train_loss = 0.221414 | val_loss = 0.475590
Epoch 090 | train_loss = 0.199006 | val_loss = 0.462268
Epoch 100 | train_loss = 0.217569 | val_loss = 0.452763
